In [28]:
import pandas as pd
from modAL.models import ActiveLearner
from modAL.models import CommitteeRegressor
from modAL.disagreement import vote_entropy_sampling
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import torch
from kan import KAN, create_dataset_from_data
from kan import *

In [64]:
df = pd.read_csv('ubend_gen_v4.csv')
for i in df.columns:
    if df[i].dtype is not np.float64:
        df[i] = df[i].astype(np.float64)
print(df.columns)
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
X_pool = train_df.drop("PT_LOSS", axis=1).values
y_pool = train_df["PT_LOSS"].values.reshape(-1, 1)
print(X_pool)
# tX, ty = torch.from_numpy(X_pool).float(), torch.from_numpy(y_pool).float()
# dataset = create_dataset_from_data(tX, ty)
# print(tX, ty)

Index(['B4TOD4', 'B5TOB4', 'R45TOB4', 'OMEGA5', 'GAMMAU', 'ALPHA4', 'RE',
       'PT_LOSS'],
      dtype='object')
[[ 1.00e-01  1.25e+00  5.00e+00 ...  5.00e-01  2.00e+01  5.00e+05]
 [ 2.00e-02  1.25e+00  5.00e+00 ...  0.00e+00  2.00e+01  1.00e+05]
 [ 1.00e-01  1.00e+00  9.00e-01 ...  0.00e+00  3.00e+01  5.00e+05]
 ...
 [ 1.00e-01  1.00e+00  2.50e+00 ...  0.00e+00  1.00e+01  1.00e+05]
 [ 2.00e-02  1.50e+00  2.50e+00 ...  5.00e-01  6.00e+01  1.00e+05]
 [ 1.00e-01  1.50e+00  9.00e-01 ... -5.00e-01  1.00e+01  1.00e+05]]


In [52]:
# Предполагаемая структура модели KAN
class KANModel(KAN):
    def __init__(self,**params):
        # инициализация параметров модели
        super().__init__(**params)
    # def train(self, X, y):
    #     # Обучение модели на данных X с ответами y
    #     print("KANModel training on data shape:", X.shape)
    #     # Реальная логика обучения модели
    #     pass
    # def infer(self, X):
    #     # Предсказания для данных X
    #     print("KANModel predicting on data shape:", X.shape)
    #     # Возвращаем предсказания, в формате который нужен (например, вероятности классов)
    #     # Для примера возвращаем случайные вероятности для 2 классов
    #     n_samples = X.shape[0]
    #     return np.random.rand(n_samples, 2)
# model = KANModel(width=[2, 2], grid=3, k=3)


# Класс-обертка чтобы соблюсти API scikit-learn для modAL
class KANWrapper:
    def __init__(self, **params):
        self.model = KANModel(**params)
    def fit(self, X, y, opt = "LBFGS", steps = 80, lamb=0.001):
        dataset = create_dataset_from_data(X, y)
        self.model.fit(dataset, opt=opt, steps=steps, lamb=lamb)
        return self
    def predict(self, X):
        # modAL ожидает predict возвращающий метки классов,
        # можно получить вероятности вызовом predict_proba аналогично
        probs = self.model(X).detach().numpy()
        # Получаем метки класса с максимальной вероятностью
        # return np.argmax(probs, axis=1)
        return probs
    def predict_proba(self, X):
        # Для методов активного обучения нужны вероятности
        return self.model.infer(X)


# if __name__ == "__main__":
#     # model = KANWrapper()
#     # Создаем активного ученика modAL с нашей моделью
#     learner = ActiveLearner(
#         estimator=model,
#         X_training=tX,
#         y_training=ty
#     )
#     # Делаем запрос следующего информативного объекта для разметки (активное обучение)
#     # X_pool = np.random.rand(100, 5)
#     query_idx, query_instance = learner.query(X_pool)
#     print("Индекс объекта для разметки:", query_idx)

checkpoint directory created: ./model
saving model version 0.0
checkpoint directory created: ./model
saving model version 0.0


In [73]:
n_members = 2 # количесво моделей
learner_list = list()
grid1 = [3,7]
k1 = [5,3]
n_queries = 10  # Количество итераций активного обучения
# for i in range(n_queries):
for member_idx in range(n_members):
    # initial training data
    n_initial = 7
    train_idx = np.random.choice(range(X_pool.shape[0]), size=n_initial, replace=False)
    print(train_idx)
    X_train = X_pool[train_idx]
    y_train = y_pool[train_idx]
    print(X_train)
    tX, ty = torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float()
    # creating a reduced copy of the data with the known instances removed
    X_pool = np.delete(X_pool, train_idx, axis=0)
    y_pool = np.delete(y_pool, train_idx)

    # initializing learner
    learner = ActiveLearner(
        estimator=KANWrapper(width=[7, 7, 7, 1], grid=grid1[member_idx], k=k1[member_idx],seed=42),     #вот сюда засовываем наш KAN
        X_training=tX, y_training=ty
    )
    learner_list.append(learner)

# assembling the committee

committee = CommitteeRegressor(learner_list=learner_list)

[644 347 803 366  35   9 456]
[[ 5.00e-03  1.00e+00  5.00e+00  0.00e+00  0.00e+00  2.00e+01  5.00e+05]
 [ 2.00e-02  1.00e+00  5.00e+00  0.00e+00 -5.00e-01  3.00e+01  1.00e+05]
 [ 2.00e-02  1.50e+00  2.50e+00  0.00e+00  0.00e+00  1.00e+01  5.00e+05]
 [ 1.00e-01  1.25e+00  5.00e+00  0.00e+00 -5.00e-01  1.00e+01  5.00e+05]
 [ 5.00e-03  1.00e+00  9.00e-01  0.00e+00  0.00e+00  4.00e+01  5.00e+05]
 [ 5.00e-03  1.00e+00  5.00e+00  5.00e+00 -5.00e-01  6.00e+01  1.00e+05]
 [ 5.00e-03  1.25e+00  2.50e+00  0.00e+00 -5.00e-01  6.00e+01  5.00e+05]]
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 1.08e-01 | test_loss: 3.05e-01 | reg: 1.18e+01 | : 100%|█| 80/80 [00:12<00:00,  6.64it


saving model version 0.1
[661 220 155 677 911 168 795]
[[ 1.00e-01  1.25e+00  2.50e+00  0.00e+00  5.00e-01  4.00e+01  1.00e+05]
 [ 5.00e-03  1.25e+00  5.00e+00  0.00e+00 -5.00e-01  4.00e+01  1.00e+05]
 [ 2.00e-02  1.25e+00  5.00e+00  0.00e+00  5.00e-01  3.00e+01  5.00e+05]
 [ 2.00e-02  1.25e+00  2.50e+00  0.00e+00  5.00e-01  4.00e+01  5.00e+05]
 [ 2.00e-02  1.50e+00  2.50e+00  0.00e+00  0.00e+00  5.00e+01  5.00e+05]
 [ 5.00e-03  1.00e+00  9.00e-01  5.00e+00  5.00e-01  3.00e+01  5.00e+05]
 [ 5.00e-03  1.50e+00  5.00e+00  0.00e+00  5.00e-01  1.00e+01  5.00e+05]]
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 1.96e-01 | test_loss: 1.21e-01 | reg: 1.93e+01 | : 100%|█| 80/80 [00:03<00:00, 25.36it

saving model version 0.1
1


In [93]:
# train_idx = np.random.choice(range(X_pool.shape[0]), size=2, replace=False)
# X_train = X_pool[train_idx]
# tX = torch.from_numpy(X_train).float()
# a = committee.predict(tX)
# print(a)
# print(X_train)
# print(X_pool)
print(y_pool[100],type(y_pool[100]))
print(X_pool[100],type(X_pool[100]))
model1 = KANWrapper(width=[7, 7, 7, 1], grid=3, k=5,seed=42)
model1.fit(tX,ty)


0.115285 <class 'numpy.float64'>
[ 5.00e-03  1.25e+00  9.00e-01 -5.00e+00  0.00e+00  1.00e+01  1.00e+05] <class 'numpy.ndarray'>
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 1.17e-01 | test_loss: 1.43e-01 | reg: 1.62e+01 | : 100%|█| 80/80 [00:17<00:00,  4.48it

saving model version 0.1


In [101]:

# tx1 = torch.from_numpy(X_pool[100]).float()
# print(tx1)
# ty1 = y_pool[100]
# pred = model1.predict(tx1)
# loss = torch.nn.MSELoss()(pred, ty1)
# loss.backward()
# # Получаем производные по входным данным
# gradients = x.grad
# print(gradients)

tensor([ 5.0000e-03,  1.2500e+00,  9.0000e-01, -5.0000e+00,  0.0000e+00,
         1.0000e+01,  1.0000e+05])


IndexError: too many indices for tensor of dimension 1

In [103]:
def qbc(committee, X_sample,y_sample):
    qsum = 0
    f_average = 0
    tx = torch.from_numpy(X_sample).float() 
    for model_ind in range(len(committee)):
        pred = committee[model_ind].predict(tx)
        f_average += pred
    f_average = f_average/len(committee)
    for model in committee:
        qsum += (model.predict(tx) - f_average)*model
    qsum /= len(committee)
    return qsum
            



    
    

In [ ]:
def NA_QBC(committee, X_pool, y_pool, n_initial):
    train_idx = np.random.choice(range(X_pool.shape[0]), size=n_initial, replace=False)
    Sbp = 2
    X_add = X_pool[train_idx]
    y_add = y_pool[train_idx]
    for x_i in range(n_initial):
        



In [ ]:
n_queries = 10  # Количество итераций активного обучения
n_committee = 5  # Количество моделей в комитете
# Основной цикл активного обучения
for i in range(n_queries):
    # Создание комитета моделей
    committee = [RandomForestRegressor() for _ in range(n_committee)]
    
    # Обучение моделей на текущем наборе данных
    for model in committee:
        model.fit(X_train, y_train)
    
    # Получение предсказаний от всех моделей
    predictions = np.array([model.predict(X_pool) for model in committee])

# Вычисление неопределенности (разброс предсказаний)
    uncertainty = np.std(predictions, axis=0)
    
    # Выбор экземпляра с наибольшей неопределенностью
    query_index = np.argmax(uncertainty)
    
    # Добавление выбранного экземпляра в обучающую выборку
    X_train = np.vstack((X_train, X_pool[query_index].reshape(1, -1)))
    y_train = np.append(y_train, y_pool[query_index])
    
    # Удаление выбранного экземпляра из пула
    X_pool = np.delete(X_pool, query_index, axis=0)
    y_pool = np.delete(y_pool, query_index)

In [ ]:
# def qbc(committee, X_pool, y_pool, n_initial):
#     qsum = [0 for i in range(len(n_initial)]
#     f_average = [0 for i in range(len(n_initial)]
#     train_idx = np.random.choice(range(X_pool.shape[0]), size=n_initial, replace=False)
#     Sbp = 2
#     X_add = X_pool[train_idx]
#     y_add = y_pool[train_idx]
#     for i in range(n_initial):
#         sample = X_add[i]
#         tx = torch.from_numpy(sample).float() 
#         for model_ind in range(len(committee)):
#             f_average[n_initial] += committee[model_ind].predict(tx)
#         f_average[n_initial] = f_average[n_initial]/len(committee)
#     for i in range(n_initial):
#         sample = X_add[i]
#         for model in committee:
#             qsum[i] += (model.predict(tx) - f_average[i])**2
#         qsum[i] /= len(committee)
#     return qsum